# 📊 Análise de Vendas e Impacto de Descontos
## 1. Contexto de negócio
Em mercados competitivos, descontos são frequentemente utilizados como mecanismo para estimular vendas, aumentar participação de mercado e acelerar o giro comercial.
Entretanto, existe uma questão estratégica relevante:
até que ponto conceder descontos contribui para o crescimento do negócio sem comprometer sua rentabilidade?
Este projeto investiga essa relação a partir de uma base transacional de vendas, buscando entender como diferentes níveis de desconto impactam faturamento, lucro, margem e ticket médio.
## 2. Objetivo da análise
O objetivo central deste estudo é avaliar se a política comercial de descontos está efetivamente gerando valor econômico para a operação.
Mais especificamente, a análise busca responder:
* Os descontos aumentam resultado financeiro ou apenas reduzem receita?
* Existe um limite a partir do qual o desconto passa a destruir margem?
* Quais regiões e categorias são mais sensíveis à política de preços?
* O comportamento se mantém consistente ao longo do tempo?
## 3. Fonte dos dados
A base utilizada neste estudo é o conjunto sales dataset, disponibilizado por VINOTH KANNA na plataforma Kaggle.
Características da base:
* 1.000 registros de vendas
* Período entre janeiro de 2023 e janeiro de 2024
* Comparação entre cenários com e sem desconto;
* Variáveis de preço, custo, desconto, categoria, região, canal e cliente
* Endereço: https://www.kaggle.com/datasets/vinothkannaece/sales-dataset

---

## 4. Bibliotecas utilizadas

In [1]:
# Bibliotecas utilizadas
import pandas as pd
#import numpy as np
#import matplotlib.pyplot as plt
import import_ipynb
from pathlib import Path
from pyprojroot import here

---

## 5. Imports

In [9]:
from sales_analysis.utils.csv_read.csv_load import *
from sales_analysis.utils.quality.quality_test import *
from sales_analysis.utils.kpis.kpis_list import *

## 6. Leitura dos dados

In [3]:
ROOT = here()
file = Path(ROOT / "data/sales_data.csv")

In [4]:
df = load_csv(file)

---

## 7. Visualização inicial dos dados

In [5]:
df.head(5)

,Product_ID,Sale_Date,Sales_Rep,Region,Sales_Amount,Quantity_Sold,Product_Category,Unit_Cost,Unit_Price,Customer_Type,Discount,Payment_Method,Sales_Channel,Region_and_Sales_Rep
0,1052,2023-02-03,Bob,North,5053.97,18,Furniture,152.75,267.22,Returning,0.09,Cash,Online,North-Bob
1,1093,2023-04-21,Bob,West,4384.02,17,Furniture,3816.39,4209.44,Returning,0.11,Cash,Retail,West-Bob
2,1015,2023-09-21,David,South,4631.23,30,Food,261.56,371.40,Returning,0.20,Bank Transfer,Retail,South-David
3,1072,2023-08-24,Bob,South,2167.94,39,Clothing,4330.03,4467.75,New,0.02,Credit Card,Retail,South-Bob
4,1061,2023-03-24,Charlie,East,3750.20,13,Electronics,637.37,692.71,New,0.08,Credit Card,Online,East-Charlie


## Observações iniciais
* O dataset contém informações sobre vendas
* Há variáveis categóricas(ex: Region), numéricas(ex: Sales_Amount) e do tipo data(ex: Sale_Date)
* Aparentemente sem valores nulos, incorretos, duplicados ou inconsistentes.

---

## 8. Estrutura dos dados
Nesta etapa, analisamos os tipos de dados, a quantidade de registros, a quantidade e nomes das colunas e memória utilizada.

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Product_ID            1000 non-null   int64  
 1   Sale_Date             1000 non-null   str    
 2   Sales_Rep             1000 non-null   str    
 3   Region                1000 non-null   str    
 4   Sales_Amount          1000 non-null   float64
 5   Quantity_Sold         1000 non-null   int64  
 6   Product_Category      1000 non-null   str    
 7   Unit_Cost             1000 non-null   float64
 8   Unit_Price            1000 non-null   float64
 9   Customer_Type         1000 non-null   str    
 10  Discount              1000 non-null   float64
 11  Payment_Method        1000 non-null   str    
 12  Sales_Channel         1000 non-null   str    
 13  Region_and_Sales_Rep  1000 non-null   str    
dtypes: float64(4), int64(2), str(8)
memory usage: 109.5 KB


## Principais pontos
* O dataset possui 1000 linhas e 14 colunas
* Variáveis numéricas estão representadas como `int64` e `float64`
* Variáveis categóricas estão como `object` (texto)
* A coluna `Sale_Date` precisará de conversão para `datetime`
* Sem valores nulos

---

## 9. Teste de qualidade dos dados
A função implementa uma série de testes para garantir a qualidade dos dados

In [7]:
get_quality_test(df)

🔍 CHECKLIST DE QUALIDADE DE DADOS

📌 Shape: (1000, 14)
📌 Valores Nulos: 0
📌 Valores Duplicados: 0
📌 Valores Negativos: 0
⚠️ Tipagem da data: str
📌 Datas nulas: 0
📌 Data mínima: 2023-01-01
📌 Data máxima: 2024-01-01
📌 Faturamento sem desconto (inconsistências): 0 (0.00%)
⚠️ Faturamento com desconto (inconsistências): 604 (60.40%)


## Resultado da validação
* A base apresentou boa consistência estrutural.
* No entanto, além da data com tipo errado, surgiu um primeiro sinal relevante:
* 60,4% das transações com desconto apresentaram preço líquido inferior ao custo unitário.
* Esse achado já sugere que a política comercial pode estar comprometendo a rentabilidade operacional.

---

## ⚠️ Nota metodológica sobre Faturamento e Sales_Amount
De acordo com o dicionário do conjunto de dados, Sales_Amount representa o valor total da venda, já considerando eventuais descontos aplicados.

Neste projeto, optou-se por não utilizar Sales_Amount como base principal das métricas de faturamento, lucro e margem. Em vez disso, foi adotada uma métrica analítica construída a partir de Quantity_Sold, Unit_Price, Unit_Cost e Discount.

Essa escolha metodológica permite analisar de forma mais direta o efeito operacional de preço, volume e desconto sobre a rentabilidade, preservando consistência interna nas métricas utilizadas ao longo da análise.

---

## 10. Conversão de tipo
A coluna data de venda (`Sale_Date`) será convertida para o tipo `datatime`

In [8]:
df["Sale_Date"] = pd.to_datetime(df["Sale_Date"], errors="coerce")

---

## 11. Criação dos principais KPIs

In [11]:
# KPIs Financeiros - Métricas base
df["faturamento_sem_desc"] = df["Quantity_Sold"] * df["Unit_Price"]
df["faturamento_com_desc"] = df["Quantity_Sold"] * df["Unit_Price"] * (1 - df["Discount"])
df["custo"] = df["Quantity_Sold"] * df["Unit_Cost"]
df["lucro_sem_desc"] = df["faturamento_sem_desc"] - df["custo"]
df["lucro_com_desc"] = df["faturamento_com_desc"] - df["custo"]
df["margem_com_desc"] = df["lucro_com_desc"] / df["faturamento_com_desc"]
df["margem_sem_desc"] = df["lucro_sem_desc"] / df["faturamento_sem_desc"]

---

## 12. Estatística

In [12]:
df.describe(include="number")

,Product_ID,Sales_Amount,Quantity_Sold,Unit_Cost,Unit_Price,Discount,faturamento_sem_desc,faturamento_com_desc,custo,lucro_sem_desc,lucro_com_desc,margem_com_desc,margem_sem_desc
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,1050.128000,5019.265230,25.355000,2475.304550,2728.440120,0.15239,70329.940710,59686.173284,63842.093640,6487.847070,-4155.920356,-0.021608,0.142757
std,29.573505,2846.790126,14.159006,1417.872546,1419.399839,0.08720,57783.299413,49696.370608,55398.598783,5444.188837,11291.253726,0.211082,0.154231
min,1001.000000,100.120000,1.000000,60.280000,167.120000,0.00000,211.180000,154.161400,60.280000,17.360000,-54935.730000,-0.402296,0.002215
25%,1024.000000,2550.297500,13.000000,1238.380000,1509.085000,0.08000,23363.535000,19710.004375,18591.320000,1892.302500,-8178.629325,-0.165445,0.048518
50%,1051.000000,5019.300000,25.000000,2467.235000,2696.400000,0.15000,54519.175000,45703.685300,48453.135000,5236.825000,-1210.456500,-0.045353,0.093993
75%,1075.000000,7507.445000,38.000000,3702.865000,3957.970000,0.23000,105855.845000,91333.039050,97755.397500,9935.082500,1874.734775,0.063047,0.163021
max,1100.000000,9989.040000,49.000000,4995.300000,5442.150000,0.30000,252147.360000,249625.886400,235808.090000,23441.110000,20832.446400,0.820913,0.841738


## Interpretação
A base analisada contém 1.000 registros de vendas, o que fornece uma amostra consistente para avaliação do desempenho comercial.

### Visão geral das vendas
O volume médio por transação é de 25,4 unidades (Quantity_Sold), com grande dispersão (std = 14,16), o que indica heterogeneidade relevante entre pedidos, coexistem vendas pequenas e grandes.

Em média, cada transação apresentou quantidade vendida de 25 unidades, com preço unitário médio de 2.728,44 e custo unitário médio de 2.475,30. O desconto médio aplicado foi de 15,24%, indicando uma política comercial relativamente frequente de concessão de descontos.

Em termos financeiros, o faturamento médio sem desconto foi de 70.329,94, enquanto o faturamento médio com desconto caiu para 59.686,17, evidenciando impacto direto das reduções de preço sobre a receita. O custo médio por operação foi de 63.842,09.

### Ao comparar os cenários, com e sem descontos, observa-se que:
* Em 50% das vendas, o lucro com desconto ficou abaixo de -1.210,46.
* Já sem desconto, a mediana do lucro foi 5.236,83.
Isso reforça que o impacto negativo dos descontos não se limita a poucos casos extremos, mas aparece de forma recorrente em boa parte das operações.

Conclusão: de forma geral, os dados indicam que a empresa possui estrutura de preço capaz de gerar lucro sem descontos, porém o nível médio de desconto atualmente praticado compromete a margem operacional, sugerindo a necessidade de revisão da política comercial ou segmentação mais criteriosa dos descontos concedidos.

---